In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
from openai import OpenAI
client = OpenAI()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [3]:
call_llm("Hello")

'Hello! How can I assist you today?'

In [4]:
import os
def load_prompt(filename):
    path = os.path.join("prompt", filename)
    with open(path) as f:
        return f.read()

In [9]:
basic_prompt = load_prompt("classify_basic.md")

In [13]:
# print(prompt)
email = {'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [11]:
print(basic_prompt)

You are an expert support email classifier
for a B2B SaaS company.

You are a senior support agent with 10 years of experience.
You know that surface words do not always reveal the real issue.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Rules:
- Login broken because of recent payment → Billing, NOT Technical
- Sarcastic complaints ("wow great job") → still Billing
- Multi-issue email → classify by ROOT CAUSE
- Auto-reply / out-of-office → Other
- Pricing complaint alongside feature requests → Billing
- If unsure → Other

Subject : {subject}
From    : {sender}
Body    : {body}

Return ONLY: Category | Confidence (1-10)



In [15]:
final_prompt =basic_prompt.format(subject=email['subject'],sender=email['sender'], body=email['body'])

In [16]:
print(final_prompt)

You are an expert support email classifier
for a B2B SaaS company.

You are a senior support agent with 10 years of experience.
You know that surface words do not always reveal the real issue.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Rules:
- Login broken because of recent payment → Billing, NOT Technical
- Sarcastic complaints ("wow great job") → still Billing
- Multi-issue email → classify by ROOT CAUSE
- Auto-reply / out-of-office → Other
- Pricing complaint alongside feature requests → Billing
- If unsure → Other

Subject : App crashes on settings page
From    : user@bigclient.com
Body    : Every time I open Settings the app crashes. Chrome 120, Windows 11.

Return ONLY: Category | Confidence (1-10)



In [19]:
response =call_llm(final_prompt)
print(response)

Technical | 9


In [21]:
cot_prompt = load_prompt("classify_cot.md")

cot_final = cot_prompt.format(subject=email['subject'],sender=email['sender'], body=email['body'])

In [24]:
email

{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [23]:
cot_response = call_llm(cot_final)
print(cot_response)

THINKING:
1. Surface complaint: The app crashes when trying to access the Settings page.
2. Root cause: Likely a technical issue with the app's compatibility or stability on the specified platform.
3. Team that OWNS this: Technical team.
4. Business impact: Medium impact, as the user can work around the issue but it affects their ability to manage settings effectively.

CATEGORY: Technical
URGENCY: Medium


In [29]:
tot=load_prompt("tree_of_thought.md")

In [34]:
email

{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [33]:
tot_prompt = tot.format(subject=email['subject'], body=email['body'],category='Technical')



In [35]:
tot_response = call_llm(tot_prompt)
print(tot_response)

**STRATEGY 1: Immediate Discount Offer**  
Response:  
Subject: We’re Here to Help with the App Crash  

Hi [Customer's Name],  

Thank you for reaching out and letting us know about the app crashing when trying to access the Settings page. We apologize for the inconvenience this has caused. To make up for this experience, we’d like to offer you a 20% discount on your next subscription renewal.  

In the meantime, I recommend clearing your browser cache or trying the app in a different browser to see if that helps. Please let us know how it goes!  

Best regards,  
[Your Name]  
Senior Customer Success Manager  

Score: 5/10  
Why: While a discount offer shows goodwill, it does not address the technical issue directly or seek to resolve the customer's problem.

---

**STRATEGY 2: Empathetic + Schedule a Call**  
Response:  
Subject: Troubleshooting Your App Issue  

Hi [Customer's Name],  

I sincerely apologize for the inconvenience caused by the app crashing on the Settings page. I u

In [ ]:
tot_prompt

In [37]:
result = """THINKING:
1. Surface complaint: User is experiencing an issue with logging into the portal.
2. Root cause: The login problem could be related to a variety of factors such as incorrect credentials, account issues, or technical failures.
3. Team that OWNS this: Technical team, as login issues typically involve troubleshooting the application or system.
4. Business impact: Medium, as while the user is unable to access the portal, it's unclear if this prevents them from completing critical tasks; there might be workarounds available.

CATEGORY: Technical
URGENCY: Medium"""

In [39]:
category, urgency = None, None
for line in result.split('\n'):
    if line.startswith('CATEGORY:'):
        category = line.split(':')[1].strip()
    elif line.startswith('URGENCY:'):
        urgency = line.split(':')[1].strip()

In [40]:
category

'Technical'

In [41]:
urgency

'Medium'

In [ ]:
category = ['Billing', 'Technical', 'Billing', 'Billing', 'Billing']

from collections import Counter
category_counts = Counter(category)
category_counts

Counter({'Billing': 5})

In [47]:
category_counts.most_common(1)[0][0]

'Billing'

In [48]:
4/5

0.8

In [49]:
1/5

0.2